# KV Cache & 추론 최적화 - 실습 코드 1: vLLM을 사용한 최적화 추론

- Tutorial ID: `expand-kv-cache`
- Tutorial: KV Cache & 추론 최적화
- Section ID: `expand-kv-cache-code-1`
- Section: 실습 코드 1: vLLM을 사용한 최적화 추론

---

> 📘 **이 노트북에 대해**
>
> 원본 실습 코드를 처음 공부하는 분도 혼자 따라갈 수 있도록, 개념 설명과 숫자로 확인하는 예제를 더해 다시 구성했습니다. 새로운 용어는 등장하기 전에 먼저 개념과 비유로 설명한 뒤 코드로 넘어갑니다.
>
> | Part | 내용 |
> |---|---|
> | 0 | 개념 준비 — KV Cache 복습, vLLM은 왜 필요한가, PagedAttention 비유, 핵심 키워드 미리보기 |
> | 1 | 실습 환경 준비 — 요구사항, 설치, GPU 확인 |
> | 2 | 모델 준비 — 모델 선택, gated 모델 로그인 |
> | 3 | vLLM 없이 돌려보기 — naive 방식 체감 (선택) |
> | 4 | 생성 파라미터 이해 — temperature/top_p/top_k를 숫자로 |
> | 5 | vLLM 엔진 초기화 — 파라미터별 설명 + KV 캐시 메모리 계산 |
> | 6 | SamplingParams 설정 |
> | 7 | 배치 추론 실행 |
> | 8 | 출력 결과 완전 분해 |
> | 9 | Prefix Caching 효과 직접 확인 |
> | 10 | naive vs vLLM 비교 정리 |
> | 11 | kv_cache_dtype 옵션 정리 (심화) |
> | 12 | 자주 겪는 오류 (Troubleshooting) |
> | 13 | 정리 + 스스로 실험해보기 + 다음 예고 |

In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: vLLM을 사용한 최적화 추론
#
# 이 노트북은 "정답 코드를 한 번 실행"하는 용도가 아니라,
# vLLM이 내부적으로 무엇을 최적화하는지, 그리고 그 효과가
# 실제 속도/메모리 숫자로 어떻게 나타나는지 직접 확인하기 위한 실습 노트입니다.
#
# 학습 목표:
#   1) vLLM이 없을 때(naive)와 있을 때, 같은 프롬프트 3개를 처리하는 시간이
#      얼마나 달라지는지 직접 측정합니다.
#   2) logit이 softmax를 거쳐 확률분포로 바뀌는 과정과, temperature/top-k/top-p가
#      그 분포를 어떻게 바꾸는지 작은 숫자 예제로 관찰합니다.
#   3) PagedAttention이 KV 캐시를 "페이지" 단위로 나눠 메모리 낭비를 줄이는 원리를
#      실제 메모리 계산으로 확인합니다.
#   4) Prefix Caching이 같은 프롬프트의 앞부분을 재사용해 계산을 줄이는 과정을
#      num_cached_tokens 값으로 직접 확인합니다.
#   5) KV 캐시를 fp8로 양자화하면 메모리가 얼마나 절약되는지 직접 계산합니다.
#
# 읽는 순서:
#   1) Part 0: 개념(왜 필요한가)을 먼저 읽습니다. 아직 코드는 없습니다.
#   2) Part 1~2: 실습 환경(GPU, 모델)을 준비합니다.
#   3) Part 3: vLLM 없이 돌려보고 "무엇이 느린지" 체감합니다. (선택 실습)
#   4) Part 4~8: 샘플링 파라미터와 vLLM 엔진을 하나씩 이해하며 실제로 추론을 실행합니다.
#   5) Part 9~11: Prefix Caching 효과, naive 대비 속도, 양자화 효과를 직접 측정합니다.
#   6) Part 12~13: 자주 만나는 오류를 확인하고, 배운 내용을 정리합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 "무엇이 왜 빨라지고, 무엇이 왜 메모리를 아끼는지"의
#     흐름을 따라가 보세요.
#   - 이 노트북의 vLLM 관련 코드 셀은 NVIDIA GPU + Linux 환경(예: Colab의 GPU 런타임)에서
#     실행해야 합니다. CPU 전용 환경이나 Mac(Apple Silicon)에서는 실행되지 않습니다.
#   - model, temperature, gpu_memory_utilization, kv_cache_dtype 같은 값을 직접 바꿔가며
#     결과가 어떻게 달라지는지 실험해보는 것을 강력히 권장합니다. (Part 13 참고)
# ============================================================

## Part 0. 시작하기 전에 — 개념 준비

### 0-1. 잠깐 복습: KV Cache가 왜 필요했을까

트랜스포머 기반 언어모델은 답을 한 토큰씩 순서대로 만들어냅니다 (이를 **autoregressive decoding**이라 부릅니다). 새 토큰을 하나 만들 때마다, 모델은 "지금까지 나온 모든 토큰"에 대한 Key(K)와 Value(V)를 다시 계산해야 할까요?

- **KV Cache가 없다면**: 토큰을 하나 생성할 때마다 이전 토큰 전체의 K/V를 처음부터 다시 계산해야 합니다. 문장이 길어질수록 매 스텝마다 반복 계산이 크게 늘어납니다.
- **KV Cache가 있다면**: 한 번 계산한 K/V를 메모리에 저장해두고, 새 토큰의 K/V만 추가로 계산해서 이어 붙입니다. 매 스텝의 계산량이 훨씬 줄어듭니다.

정리하면, KV Cache는 "**계산 시간**"을 아끼기 위한 장치입니다. 그런데 이 장치에는 대가가 있습니다 — 바로 "**메모리**"입니다. 문장이 길어지고, 동시에 처리하는 사용자가 많아질수록 저장해둬야 하는 K/V의 양도 함께 늘어납니다.

이번 실습은 바로 이 "메모리 대가"를 실제 서비스 환경에서 얼마나 똑똑하게 관리할 수 있는지에 대한 이야기입니다.

### 0-2. 그런데 왜 또 최적화가 필요할까? — naive 서빙의 한계

KV Cache 자체는 이미 "한 문장을 생성하는" 속도를 크게 높여줍니다. 문제는 **여러 사용자의 요청을 동시에** 처리해야 하는 실제 서비스 상황에서 발생합니다. 흔히 아래와 같은 비효율이 생깁니다.

1. **요청을 한 번에 하나씩만 처리 (배치 처리 부재)**
   사용자 A, B, C가 거의 동시에 질문했는데, A의 답변이 다 끝날 때까지 B와 C는 순서를 기다립니다. GPU는 원래 여러 계산을 한꺼번에 하는 데 강한 하드웨어인데, 이런 방식으로는 그 장점을 거의 살리지 못합니다.
2. **메모리를 최대 길이 기준으로 미리 통째로 예약**
   "혹시 문장이 아주 길어질 수도 있으니" 하고 각 요청마다 최대 길이만큼 KV 캐시 공간을 미리 잡아두면, 실제로는 짧게 끝나는 요청이 대부분이라도 예약된 메모리는 그대로 낭비됩니다.
3. **같은 내용을 반복 계산**
   여러 사용자가 같은 시스템 프롬프트(예: "당신은 친절한 상담원입니다...")를 공유하는데도, 매번 그 부분을 처음부터 다시 계산합니다.

세 문제 모두 "계산 방법 자체(수학)"가 아니라 "**어떻게 메모리와 스케줄링을 관리할 것인가**"에 대한 문제입니다. **vLLM**은 바로 이 지점을 해결하기 위해 만들어진 추론 서빙 엔진입니다.

### 0-3. vLLM 소개 + PagedAttention 비유

**vLLM**은 UC Berkeley에서 시작된 오픈소스 프로젝트로, LLM을 "빠르고, 메모리 효율적으로, 많은 사용자에게 동시에" 서빙하기 위한 추론 엔진입니다. vLLM의 핵심 아이디어는 **PagedAttention**이라는 기법입니다.

**비유로 이해하기: 호텔 객실 예약**

- **기존 방식(전통적인 KV 캐시 관리)**: 손님(요청)이 예약하면, "혹시 오래 머물지도 모르니" 가장 큰 스위트룸(최대 길이만큼의 연속된 메모리)을 통째로 비워둡니다. 손님이 하루만 자고 나가도 그 방은 이미 이 손님 전용으로 예약되어 있었기 때문에 다른 손님이 쓸 수 없습니다. → 메모리 낭비(내부 단편화)가 심각합니다.
- **PagedAttention 방식**: 객실을 작은 표준 크기의 "블록(페이지)" 단위로 쪼갭니다. 손님이 필요한 만큼만 그때그때 블록을 배정하고, 손님이 나가면 그 블록을 즉시 회수해서 다른 손님에게 재배정합니다. 심지어 여러 손님이 같은 블록(같은 프롬프트 앞부분)을 공유해서 쓸 수도 있습니다.

이 아이디어는 운영체제(OS)가 프로그램에게 물리 메모리를 "페이지" 단위로 나누어 주는 **가상 메모리(virtual memory paging)** 방식과 거의 동일합니다. vLLM은 이 아이디어를 GPU의 KV 캐시 관리에 그대로 적용했습니다. 그 결과 메모리 낭비를 크게 줄이고, 그만큼 확보된 메모리로 더 많은 요청을 동시에 처리할 수 있게 됩니다.

### 0-4. 이 실습에서 만날 핵심 키워드 4가지 (미리보기)

아래 4가지는 이번 실습 코드에서 실제로 등장하는 vLLM의 핵심 기능입니다. 지금은 표로 가볍게 훑어보고, 각 항목이 코드에 실제로 등장하는 순간 다시 자세히 설명합니다.

| 키워드 | 한 줄 설명 | 아끼는 것 |
|---|---|---|
| **PagedAttention** | KV 캐시를 작은 "블록" 단위로 나눠 필요한 만큼만 할당 | 메모리(공간) |
| **Continuous Batching** | 요청이 끝나는 즉시 새 요청을 빈 자리에 끼워 넣는 스케줄링 | GPU 대기 시간 |
| **Prefix Caching** | 여러 요청이 공유하는 프롬프트 앞부분의 K/V를 한 번만 계산 | 중복 계산 |
| **KV Cache Quantization** | K/V 값을 더 적은 비트(예: fp8)로 저장 | 메모리(용량) |

이 중 PagedAttention과 Continuous Batching은 `LLM(...)`을 생성하는 순간 vLLM이 **자동으로** 적용해주는 기본 동작이고, Prefix Caching과 Quantization은 우리가 파라미터로 켜고 끌 수 있는 옵션입니다. 아래 실습에서 하나씩 만나보겠습니다.

## Part 1. 실습 환경 준비

### 1-1. vLLM 실행 요구사항

vLLM은 아래 환경에서 동작합니다. 시작하기 전에 꼭 확인하세요.

- **운영체제**: Linux (Google Colab, 대부분의 클라우드 GPU 서버 포함)
- **하드웨어**: NVIDIA GPU (CUDA 지원). CPU 전용 환경이나 Mac(Apple Silicon, M1/M2/M3 등)에서는 이번 실습의 vLLM 코드 셀이 정상적으로 실행되지 않습니다.
- **권장**: Google Colab을 쓴다면 상단 메뉴에서 `런타임 → 런타임 유형 변경 → 하드웨어 가속기: GPU`를 먼저 선택하세요.

### 1-2. 설치

아래 셀에서 vLLM과 필요한 패키지를 설치합니다. 처음 설치할 때는 관련 라이브러리(PyTorch 등)를 함께 받기 때문에 몇 분 정도 걸릴 수 있습니다.

In [ ]:
# vLLM 설치 (처음 실행 시 몇 분 정도 소요될 수 있습니다)
# 이미 설치되어 있다면 이 셀은 건너뛰어도 됩니다.
!pip install -q vllm

# naive(HuggingFace transformers) 비교 실습(Part 3)을 위해 함께 설치합니다.
!pip install -q transformers accelerate

In [ ]:
# 본격적으로 모델을 불러오기 전에, GPU를 제대로 인식하고 있는지 먼저 확인합니다.
# 이 확인을 건너뛰고 바로 모델을 로드하면, 나중에 원인을 알기 어려운 오류를 만날 수 있습니다.

import torch

print(f"CUDA(GPU) 사용 가능 여부: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU 이름: {torch.cuda.get_device_name(0)}")
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU 총 메모리: {total_mem_gb:.1f} GB")
else:
    print("경고: GPU를 찾지 못했습니다.")
    print("Colab이라면: 런타임 -> 런타임 유형 변경 -> 하드웨어 가속기: GPU 를 선택한 뒤 다시 실행하세요.")

## Part 2. 모델 준비하기

### 2-1. 어떤 모델을 쓸까?

원본 실습 코드는 `meta-llama/Meta-Llama-3-8B-Instruct` 모델을 사용합니다. 성능은 훌륭하지만, 처음 실습하는 입장에서는 아래 두 가지 걸림돌이 있을 수 있습니다.

1. **Gated 모델**: Meta의 라이선스 조건에 동의하고 Hugging Face 계정으로 로그인해야만 다운로드할 수 있습니다.
2. **큰 메모리 요구량**: 80억(8B) 개 파라미터 모델은 fp16 기준으로 가중치만 약 16GB를 차지합니다. 여기에 KV 캐시, 연산 중 임시 메모리(activation)까지 더해지므로, GPU 메모리가 24GB 이상은 되어야 여유롭게 실습할 수 있습니다. (Colab 무료 T4는 16GB라 빠듯합니다.)

그래서 이 실습에서는 **가볍고 로그인 없이 바로 쓸 수 있는 모델을 기본값**으로 사용하고, 원본 모델은 "GPU 메모리가 넉넉한 경우의 옵션"으로 남겨둡니다. vLLM을 다루는 코드 자체는 모델이 무엇이든 동일합니다.

### 2-2. Gated 모델(예: Llama 3)을 쓰고 싶다면

1. https://huggingface.co 에 가입 후 로그인합니다.
2. 모델 페이지(예: `meta-llama/Meta-Llama-3-8B-Instruct`)에서 라이선스에 동의(`Agree and access repository`)합니다.
3. https://huggingface.co/settings/tokens 에서 Read 권한 Access Token을 발급받습니다.
4. 아래 코드 셀에서 로그인하거나, 터미널에서 `huggingface-cli login`을 실행합니다.

In [ ]:
# -----------------------------------------------------------------
# 실습에 사용할 모델을 변수 하나로 관리합니다.
# 아래 두 옵션 중 하나를 선택하세요. (기본값: 옵션 A)
# -----------------------------------------------------------------

# [옵션 A] 기본값 (추천): 가볍고 로그인 없이 바로 사용 가능
#   - Apache-2.0 라이선스 (별도 승인 불필요)
#   - fp16 기준 가중치 약 3GB -> Colab 무료 T4(16GB)에서도 여유롭게 실습 가능
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# [옵션 B] 원본 실습 코드가 사용하던 모델
#   - Hugging Face 라이선스 동의 + 로그인(access token) 필요 (2-2 참고)
#   - GPU 메모리 24GB 이상 권장
#   - 사용하려면 아래 줄의 주석(#)을 지우고, 위 옵션 A 줄은 주석 처리하세요.
# MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"

# naive 방식(Part 3)과 vLLM 방식(Part 6)을 "공정하게" 비교하기 위해,
# 최대 생성 토큰 수를 변수로 따로 빼두고 두 방식 모두에서 동일하게 사용합니다.
MAX_NEW_TOKENS = 200

print(f"이번 실습에서 사용할 모델: {MODEL_NAME}")
print(f"요청당 최대 생성 토큰 수: {MAX_NEW_TOKENS}")

# Gated 모델(옵션 B)을 쓰는 경우에만 아래 로그인 코드가 필요합니다.
# Qwen처럼 공개된 모델만 쓴다면 아래는 그대로 두어도 됩니다.

# from huggingface_hub import login
# login(token="hf_여기에_발급받은_토큰을_붙여넣으세요")

## Part 3. vLLM 없이 돌려보기 — 문제 상황 체감하기

이론만 보면 "vLLM이 빠르다"는 말이 잘 와닿지 않을 수 있습니다. 그래서 먼저 vLLM 없이, 가장 기본적인 방식(HuggingFace `transformers`의 `generate()`)으로 같은 작업을 해보고 시간을 재보겠습니다. 이렇게 하면 이후 Part 10에서 vLLM과 정직하게 비교할 수 있는 기준(baseline)이 생깁니다.

> **메모리 관련 주의사항**
> 하나의 노트북(커널)에서 `transformers`로 모델을 한 번 로드한 뒤, 이어서 vLLM으로 또 모델을 로드하면 GPU 메모리가 겹쳐서 `CUDA out of memory` 오류가 날 수 있습니다.
>
> - 이 섹션(Part 3)은 **선택 실습**입니다. 시간 비교 없이 바로 vLLM 실습(Part 4~)으로 넘어가도 무방합니다.
> - 두 방식을 모두 실행해보고 싶다면, 이 섹션을 실행한 뒤 **커널(런타임)을 재시작**하고 Part 2(모델 설정)부터 다시 실행한 다음 Part 4로 넘어가는 것을 권장합니다.
> - 재시작이 번거롭다면, 아래 셀 실행 후 `del model`과 `torch.cuda.empty_cache()`로 메모리를 정리하는 방법도 있습니다. (완전히 회수되지 않을 수도 있어 재시작이 더 안전합니다.)

In [ ]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,   # 메모리 절약을 위해 fp16으로 로드
).to("cuda")

prompts = [
    "Explain quantum computing:",
    "What is machine learning?",
    "Describe neural networks:",
]

naive_outputs = []
naive_start = time.time()

for prompt in prompts:
    # ---- 이 방식이 "naive"인 이유 ----
    # 프롬프트 3개를 for문으로 하나씩, 순서대로 처리합니다.
    # 즉, 첫 번째 문장이 끝날 때까지 두 번째 문장은 시작조차 하지 않습니다.
    # (vLLM처럼 여러 요청을 한꺼번에 GPU에 태우는 "배치 처리"가 없습니다)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=0.7,
        do_sample=True,
    )
    text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    naive_outputs.append(text)

naive_elapsed = time.time() - naive_start

print(f"[naive 방식] 프롬프트 {len(prompts)}개 처리 시간: {naive_elapsed:.2f}초\n")
for i, text in enumerate(naive_outputs):
    print(f"--- 결과 {i+1} ---")
    print(text[:150] + "...")
    print()

### naive 결과가 말해주는 것

실행 시간을 보면, "프롬프트 3개 처리 시간"이 대략 "프롬프트 1개를 생성하는 시간 × 3"에 가깝다는 것을 알 수 있습니다. GPU는 원래 수많은 연산을 병렬로 처리하는 데 강한 하드웨어인데, 이 방식에서는 그 능력을 거의 쓰지 못하고 있는 것입니다.

질문이 3개가 아니라 300개, 3000개라면 어떻게 될까요? naive 방식의 처리 시간은 거의 그대로 비례해서 늘어나지만, 곧 살펴볼 vLLM은 continuous batching 덕분에 훨씬 완만하게 늘어납니다. Part 10에서 두 방식의 시간을 나란히 정리해보겠습니다.

## Part 4. 생성 파라미터 이해하기 — temperature를 숫자로 보기

vLLM에 프롬프트를 넣기 전에, "모델이 다음 단어를 어떻게 고르는지"를 결정하는 파라미터들을 먼저 이해하고 넘어가겠습니다. 이 개념은 vLLM만의 것이 아니라 거의 모든 언어모델 생성에 공통으로 적용되는 개념입니다.

### 배경: logit → 확률

모델은 다음 토큰 후보 각각에 대해 **logit**이라는 원점수(raw score)를 출력합니다. 이 점수를 그대로 쓰지 않고, **softmax** 함수를 거쳐 "합이 1이 되는 확률분포"로 바꾼 뒤, 그 확률에 따라 다음 토큰을 하나 뽑습니다.

**temperature**는 이 softmax를 적용하기 **직전에** logit을 얼마로 나눌지 정하는 값입니다.

- temperature가 **작을수록**(예: 0.2): 원래 점수 차이가 더 크게 벌어져서, 가장 점수가 높은 후보 쪽으로 확률이 쏠립니다. → 더 확신에 찬, 예측 가능한 출력
- temperature가 **클수록**(예: 1.5): 점수 차이가 완만해져서, 여러 후보의 확률이 비슷해집니다. → 더 다양하고 무작위에 가까운 출력
- temperature = 0: 사실상 가장 높은 점수의 후보만 100% 선택 (= greedy decoding)

말로만 들으면 추상적이니, 아래 코드로 아주 작은 예제(후보 단어 4개)를 직접 계산해서 눈으로 확인해보겠습니다.

In [ ]:
import numpy as np

# 예시 상황: "다음 단어" 후보가 4개이고, 모델이 계산한 logit(원점수)이 아래와 같다고 합시다.
words = ["고양이", "강아지", "물고기", "자동차"]
logits = np.array([2.0, 1.5, 0.5, 0.1])

def softmax_with_temperature(logits, temperature):
    # logit을 temperature로 나눈 뒤 softmax를 적용합니다.
    scaled = logits / temperature
    exp = np.exp(scaled - np.max(scaled))  # 오버플로 방지를 위해 최댓값을 빼고 계산
    return exp / exp.sum()

for T in [0.2, 1.0, 1.5]:
    probs = softmax_with_temperature(logits, T)
    print(f"temperature = {T}")
    for w, p in zip(words, probs):
        print(f"  {w:5s}: {p*100:5.2f}%")
    print()

# 관찰 포인트:
#  - T=0.2 일 때: "고양이"의 확률이 90% 이상으로 압도적입니다. (거의 확정적인 선택)
#  - T=1.0 일 때: logit 그대로의 상대적 비율이 유지됩니다. (표준 상태)
#  - T=1.5 일 때: 4개 후보의 확률 격차가 눈에 띄게 줄어듭니다. (더 다양한 선택 가능)

### top_p, top_k — 확률이 낮은 후보를 걸러내는 또 다른 방법

temperature는 분포의 "모양(뾰족한 정도)"을 바꾸지만, 후보를 아예 제외하지는 않습니다. 반면 `top_p`와 `top_k`는 확률이 너무 낮은 후보를 **아예 선택지에서 제외**해서, 엉뚱하거나 부자연스러운 토큰이 뽑히는 것을 막습니다.

- **top_k**: 확률이 높은 순으로 **정확히 k개**만 후보로 남깁니다. 예를 들어 `top_k=2`라면, 위 예제에서는 "고양이", "강아지"만 후보로 남고 나머지는 제외됩니다. 남은 후보들의 확률은 합이 1이 되도록 다시 정규화합니다.
- **top_p** (nucleus sampling): 확률이 높은 순으로 후보를 누적해서 더하다가, **누적 확률이 p를 넘는 순간까지**만 후보로 남깁니다. 예를 들어 `top_p=0.9`라면, 누적 확률이 90%를 넘을 때까지 후보를 하나씩 추가합니다. top_k와 달리, 후보 개수가 상황(분포가 뾰족한지 완만한지)에 따라 유동적으로 정해집니다.

아래 코드로 앞서 만든 temperature=1.0 확률분포에 두 방법을 각각 적용해서, 어떤 후보가 남고 어떤 후보가 제외되는지 직접 확인해보겠습니다.

In [ ]:
def apply_top_p(words, probs, p):
    # 확률이 높은 순서로 정렬합니다.
    order = np.argsort(-probs)
    sorted_words = [words[i] for i in order]
    sorted_probs = probs[order]

    # 누적 확률을 계산해서, p를 처음 넘는 지점까지만 남깁니다.
    cumulative = np.cumsum(sorted_probs)
    cutoff = np.searchsorted(cumulative, p) + 1

    kept_words = sorted_words[:cutoff]
    kept_probs = sorted_probs[:cutoff]
    renormalized = kept_probs / kept_probs.sum()  # 남은 후보들의 확률 합이 다시 1이 되도록 조정
    return kept_words, renormalized

def apply_top_k(words, probs, k):
    order = np.argsort(-probs)
    kept_words = [words[i] for i in order[:k]]
    kept_probs = probs[order[:k]]
    renormalized = kept_probs / kept_probs.sum()
    return kept_words, renormalized

probs_T1 = softmax_with_temperature(logits, temperature=1.0)

print("=== top_p = 0.9 적용 결과 ===")
kept_words, kept_probs = apply_top_p(words, probs_T1, p=0.9)
for w, p in zip(kept_words, kept_probs):
    print(f"  {w:5s}: {p*100:5.2f}%  (후보로 유지)")
print(f"  제외된 후보: {set(words) - set(kept_words)}")

print()
print("=== top_k = 2 적용 결과 ===")
kept_words, kept_probs = apply_top_k(words, probs_T1, k=2)
for w, p in zip(kept_words, kept_probs):
    print(f"  {w:5s}: {p*100:5.2f}%  (후보로 유지)")
print(f"  제외된 후보: {set(words) - set(kept_words)}")

# 참고: vLLM SamplingParams의 기본값은 top_p=1.0(모든 후보 허용), top_k=-1(제한 없음)입니다.
# 즉 기본값 상태에서는 temperature만으로 분포를 조절하고, top_p/top_k는 꺼져 있는 셈입니다.

## Part 5. vLLM 엔진 초기화하기

이제 실제로 vLLM을 사용해봅니다. vLLM에서 가장 먼저 만드는 것은 `LLM` 클래스의 객체입니다. 이 객체를 생성하는 순간, 아래 세 가지가 내부적으로 한꺼번에 준비됩니다.

1. 지정한 모델의 가중치를 GPU 메모리로 로드
2. PagedAttention 기반 KV 캐시 관리자 초기화 (Part 0-3에서 본 "블록 단위 객실 관리" 시스템)
3. 여러 요청을 이어받아 처리할 스케줄러(continuous batching) 준비

아래 코드에서 각 파라미터가 무엇을 의미하는지 주석으로 하나씩 설명합니다.

In [ ]:
from vllm import LLM, SamplingParams
import time

# -----------------------------------------------------------------
# vLLM의 핵심 클래스인 LLM 객체를 생성합니다.
# 이 셀은 모델을 실제로 GPU에 올리기 때문에, 모델 크기에 따라
# 수십 초 ~ 몇 분 정도 걸릴 수 있습니다. (최초 1회만 오래 걸립니다)
# -----------------------------------------------------------------
llm = LLM(
    model=MODEL_NAME,

    # GPU 메모리 중 몇 %까지 vLLM이 점유할지 정합니다. (기본값 0.9)
    # 이 안에서 "모델 가중치 + 연산 중 임시 메모리(activation) + KV 캐시"를
    # 모두 감당해야 합니다.
    #   - 너무 낮으면(예: 0.5): KV 캐시 공간이 부족 -> 동시 처리 가능한 문장 수 감소
    #   - 너무 높으면(예: 0.99): 다른 프로세스가 쓸 메모리가 없어 OOM(메모리 부족) 위험
    # (바로 다음 셀에서 이 숫자가 왜 중요한지 실제 메모리 계산으로 확인합니다)
    gpu_memory_utilization=0.90,

    # 스케줄러가 "동시에" 붙잡고 처리할 수 있는 시퀀스(요청)의 최대 개수.
    # 비유: 식당 테이블이 256개면, 한 번에 앉을 수 있는 손님은 최대 256명입니다.
    # 값을 무작정 키운다고 빨라지는 건 아니고, GPU 메모리(KV 캐시)가
    # 감당할 수 있는 만큼만 실제로 의미가 있습니다.
    max_num_seqs=256,

    # Prefix Caching(Part 0-4에서 미리 본 키워드) 활성화 여부입니다.
    # 여러 요청이 앞부분(prefix)을 공유하면(예: 같은 시스템 프롬프트),
    # 그 부분의 K/V를 한 번만 계산해두고 재사용합니다.
    # 참고: 최신 버전의 vLLM은 이 옵션이 기본값으로도 True이지만,
    # 무엇을 켜는 옵션인지 명확히 드러나도록 이 실습에서는 직접 명시했습니다.
    # (효과는 Part 9에서 직접 측정합니다)
    enable_prefix_caching=True,

    # KV 캐시를 GPU 메모리에 저장할 때 쓸 데이터 타입(정밀도)입니다.
    #   - "auto": 모델과 같은 정밀도 사용 (보통 fp16/bf16, 값 1개당 2바이트)
    #   - "fp8" : 값 1개당 1바이트만 사용 -> KV 캐시 메모리 사용량이 절반으로 감소
    # 대가는 아주 미세한 정밀도 손실이며, 대부분의 경우 생성 품질 차이는 거의 없습니다.
    # (다음 셀에서 fp16 대비 fp8이 실제로 몇 GB를 아끼는지 계산해봅니다)
    kv_cache_dtype="fp8",
)

print("vLLM 엔진 초기화 완료!")

### gpu_memory_utilization이 왜 중요한지 — KV 캐시 메모리를 직접 계산해보기

"왜 KV 캐시 메모리 관리가 이렇게까지 중요한가?"에 대한 감을 잡기 위해, 실제 숫자로 계산해보겠습니다. 원본 실습 코드가 사용하던 `Meta-Llama-3-8B-Instruct`를 기준으로 계산합니다. (계산 방법 자체는 어떤 모델이든 동일하게 적용됩니다.)

토큰 1개를 저장하기 위한 KV 캐시 크기는 아래 공식으로 구합니다.

```
KV 캐시 크기(토큰 1개) = 2(K, V) x 레이어 수 x KV head 수 x head 차원 x (값 1개당 바이트 수)
```

- `2`: Key와 Value를 각각 따로 저장해야 하므로
- **레이어 수(num_hidden_layers)**: 32 (Llama-3-8B 기준)
- **KV head 수(num_key_value_heads)**: 8 (참고: 이 모델은 GQA(Grouped-Query Attention) 구조라, attention head 32개보다 KV head 수가 더 적습니다)
- **head 차원(head_dim)**: 128 (hidden_size 4096 ÷ attention head 32개)
- **값 1개당 바이트 수**: fp16이면 2바이트, fp8이면 1바이트

아래 코드로 직접 계산해보겠습니다.

In [ ]:
# Meta-Llama-3-8B-Instruct의 실제 설정값 (Hugging Face config.json 기준)
num_layers = 32       # 트랜스포머 블록(레이어) 개수
num_kv_heads = 8       # Key/Value에 사용되는 head 개수 (GQA 구조)
head_dim = 128          # head 하나의 차원 (hidden_size 4096 / attention head 32개)

def kv_cache_bytes_per_token(num_layers, num_kv_heads, head_dim, bytes_per_value):
    # Key용, Value용을 각각 저장해야 하므로 2를 곱합니다.
    return 2 * num_layers * num_kv_heads * head_dim * bytes_per_value

fp16_bytes = kv_cache_bytes_per_token(num_layers, num_kv_heads, head_dim, bytes_per_value=2)
fp8_bytes  = kv_cache_bytes_per_token(num_layers, num_kv_heads, head_dim, bytes_per_value=1)

print(f"토큰 1개당 KV 캐시 크기 (fp16): {fp16_bytes:,} bytes = {fp16_bytes/1024:.1f} KB")
print(f"토큰 1개당 KV 캐시 크기 (fp8) : {fp8_bytes:,} bytes = {fp8_bytes/1024:.1f} KB")

# 시나리오 1: 사용자 1명이 8,192 토큰짜리(긴 문서 분량) 대화를 하고 있다면?
ctx_len = 8192
print()
print(f"[시나리오 1] 문맥 길이 {ctx_len:,} 토큰, 사용자 1명")
print(f"  fp16 KV 캐시: {fp16_bytes * ctx_len / (1024**2):.0f} MB")
print(f"  fp8  KV 캐시: {fp8_bytes  * ctx_len / (1024**2):.0f} MB")

# 시나리오 2: 사용자 100명이 동시에 접속해서, 각자 평균 2,000 토큰 분량 대화를 하고 있다면?
num_users = 100
tokens_per_user = 2000
total_fp16_gb = fp16_bytes * tokens_per_user * num_users / (1024**3)
total_fp8_gb  = fp8_bytes  * tokens_per_user * num_users / (1024**3)

print()
print(f"[시나리오 2] 동시 사용자 {num_users}명 x 평균 {tokens_per_user:,} 토큰")
print(f"  필요한 KV 캐시 총량 (fp16): 약 {total_fp16_gb:.1f} GB")
print(f"  필요한 KV 캐시 총량 (fp8) : 약 {total_fp8_gb:.1f} GB  (fp16 대비 절반)")

**결과 해석**: 사용자 한 명이 8,192 토큰짜리 긴 대화를 나누는 것만으로도 KV 캐시에 fp16 기준 약 1GB가 필요합니다. 여기에 사용자가 100명으로 늘고 각자 평균 2,000토큰만 대화해도, 필요한 KV 캐시 총량은 fp16 기준 20GB를 훌쩍 넘습니다 — 웬만한 GPU 한 장의 전체 메모리에 맞먹거나 넘어서는 수준입니다.

`kv_cache_dtype="fp8"`을 쓰면 이 부담을 절반으로 줄일 수 있고, 그만큼 `gpu_memory_utilization`이 허용하는 범위 안에서 더 많은 사용자를 동시에 받을 수 있습니다. 이것이 `gpu_memory_utilization`과 `kv_cache_dtype`이 vLLM 설정에서 실질적으로 중요한 이유입니다.

## Part 6. SamplingParams 설정하기 (실전 코드)

Part 4에서 익힌 temperature/top_p/top_k를 이제 실제 vLLM 코드로 그대로 옮겨보겠습니다. `SamplingParams`는 "이번 생성에서 다음 토큰을 어떻게 고를지"를 요청 단위로 지정하는 객체입니다.

참고로 `LLM(...)`이 "엔진 자체의 설정"이라면, `SamplingParams`는 "요청 하나하나의 생성 설정"이라는 차이가 있습니다. 같은 엔진(`llm`)에 서로 다른 `SamplingParams`를 가진 요청을 여러 개 보낼 수도 있습니다.

In [ ]:
# Part 4에서 익힌 개념을 그대로 vLLM 코드로 옮깁니다.
sampling_params = SamplingParams(
    temperature=0.7,            # 0에 가까울수록 결정적, 높을수록 다양한 출력 (Part 4 참고)
    top_p=0.9,                  # 누적 확률 90% 안의 후보만 고려 (기본값은 1.0 = 제한 없음)
    top_k=50,                   # 확률 상위 50개 후보만 고려 (기본값은 -1 = 제한 없음)
    max_tokens=MAX_NEW_TOKENS,  # 생성할 최대 토큰 수 (naive 실습과 동일한 값으로 공정하게 비교)
)

print(sampling_params)

## Part 7. 배치 추론 실행하기

이제 원본 실습 코드의 핵심 부분입니다. 프롬프트 여러 개를 **리스트로 한 번에** `llm.generate()`에 넘깁니다. naive 방식(Part 3)처럼 for문으로 하나씩 넘기지 않는다는 점에 주목하세요. vLLM은 내부적으로 continuous batching 스케줄러가 이 요청들을 동시에 GPU에 태워 처리합니다.

In [ ]:
prompts = [
    "Explain quantum computing:",
    "What is machine learning?",
    "Describe neural networks:",
]

vllm_start = time.time()

# 프롬프트 리스트를 "한 번에" 넘깁니다. vLLM 스케줄러가 내부에서
# continuous batching으로 이 요청들을 동시에 처리합니다.
outputs = llm.generate(prompts, sampling_params)

vllm_elapsed = time.time() - vllm_start

print(f"[vLLM] 프롬프트 {len(prompts)}개 처리 시간: {vllm_elapsed:.2f}초")

## Part 8. 출력 결과 완전 분해

`llm.generate()`가 반환하는 `outputs`는 `RequestOutput` 객체의 리스트입니다. (프롬프트 1개당 `RequestOutput` 1개) 각 필드가 무엇을 담고 있는지 하나씩 뜯어보겠습니다.

- `output.prompt`: 원래 입력했던 프롬프트 문자열
- `output.outputs`: **완성된 답변 후보들의 리스트**입니다. `SamplingParams(n=1)`이 기본값이라 보통 원소가 1개뿐이라 `[0]`으로 꺼냅니다. (`n`을 2 이상으로 주면, 같은 프롬프트에 대해 서로 다른 답변 여러 개를 한 번에 받을 수 있습니다.)
- `output.outputs[0].text`: 실제로 생성된 텍스트
- `output.outputs[0].token_ids`: 생성된 토큰 ID들의 리스트
- `output.outputs[0].finish_reason`: 생성이 왜 멈췄는지. `"stop"`(자연스럽게 문장이 끝남 또는 종료 토큰 등장) 또는 `"length"`(max_tokens에 도달해서 강제로 멈춤)
- `output.num_cached_tokens`: 이번 요청에서 **Prefix Cache 덕분에 다시 계산하지 않고 재사용한 토큰 수**입니다. (Part 9에서 이 값을 직접 활용합니다)

In [ ]:
total_generated_tokens = 0

for output in outputs:
    print(f"프롬프트: {output.prompt[:50]}...")

    # outputs는 "완성된 답변 후보들의 리스트"입니다. 기본값(n=1)이라 [0]으로 꺼냅니다.
    completion = output.outputs[0]

    print(f"생성된 텍스트: {completion.text[:100]}...")
    print(f"생성된 토큰 개수: {len(completion.token_ids)}")
    print(f"종료 이유(finish_reason): {completion.finish_reason}")
    print(f"Prefix Cache로 재사용된 토큰 수: {output.num_cached_tokens}")
    print()

    total_generated_tokens += len(completion.token_ids)

throughput = total_generated_tokens / vllm_elapsed
print(f"총 생성 토큰: {total_generated_tokens}개")
print(f"처리량(throughput): 초당 약 {throughput:.1f} 토큰")

위 3개의 프롬프트는 서로 겹치는 부분이 거의 없어서 `num_cached_tokens`가 작게(또는 0에 가깝게) 나올 수 있습니다. Prefix Caching의 효과를 제대로 보려면 여러 요청이 "공유하는 긴 앞부분"이 있어야 합니다 — 바로 다음 Part 9에서 이 상황을 일부러 만들어 확인해보겠습니다.

## Part 9. Prefix Caching 효과 직접 확인하기

이번에는 일부러 "여러 요청이 앞부분을 공유하는" 상황을 만들어서, Prefix Caching이 실제로 동작하는 모습을 확인해보겠습니다.

**실험 설계**: 길고 동일한 "시스템 프롬프트"를 공유하는 질문 2개를 순서대로 보냅니다. 첫 번째 요청은 이 긴 프롬프트를 처음 계산하므로 캐시가 없고, 두 번째 요청은 앞부분이 완전히 동일하므로 캐시를 재사용할 수 있어야 합니다.

또 하나, `max_tokens`를 일부러 짧게 설정합니다. 그 이유는 —

> Prefix Caching은 프롬프트를 "읽는" 단계(prefill)만 빠르게 해줄 뿐, 새 토큰을 하나씩 "생성하는" 단계(decoding)는 똑같이 걸립니다. 만약 생성 길이가 아주 길다면, 전체 시간에서 decoding이 차지하는 비중이 커져서 prefill이 빨라진 효과가 상대적으로 잘 안 보이게 됩니다. 그래서 이번 실험에서는 `max_tokens`를 짧게 잡아 prefill 비중을 키웁니다.

In [ ]:
# 여러 요청이 공유할 "긴 프리픽스"를 만듭니다.
# (실제 서비스라면 시스템 프롬프트, 긴 문서, 대화 기록 등이 여기에 해당합니다)
shared_prefix = (
    "당신은 친절하고 정확한 AI 어시스턴트입니다. "
    "사용자의 질문에 대해 핵심을 먼저 말하고, 그 다음 근거를 간단히 덧붙이는 방식으로 답하세요. "
    "확실하지 않은 정보에 대해서는 추측하지 말고 모른다고 답하세요. "
    "전문 용어를 쓸 때는 처음 등장할 때 한 번 쉬운 말로 풀어서 설명하세요. "
    "답변은 항상 존댓말로, 정중하고 간결하게 작성하세요.\n\n"
    "아래는 사용자의 질문입니다.\n"
)

question_a = shared_prefix + "질문: 인공지능이란 무엇인가요?"
question_b = shared_prefix + "질문: 딥러닝과 머신러닝의 차이는 무엇인가요?"

# max_tokens을 짧게 잡아 prefill(프롬프트를 읽는 단계) 비중을 키웁니다.
short_params = SamplingParams(temperature=0.7, max_tokens=20)

print("--- 요청 1: 아직 캐시할 것이 없는 상태 ---")
t0 = time.time()
out_a = llm.generate([question_a], short_params)
t1 = time.time()
print(f"소요 시간: {t1 - t0:.3f}초")
print(f"Prefix Cache로 재사용된 토큰 수: {out_a[0].num_cached_tokens}")

print()
print("--- 요청 2: 앞부분(shared_prefix)이 요청 1과 동일! ---")
t2 = time.time()
out_b = llm.generate([question_b], short_params)
t3 = time.time()
print(f"소요 시간: {t3 - t2:.3f}초")
print(f"Prefix Cache로 재사용된 토큰 수: {out_b[0].num_cached_tokens}")

### 결과 해석 + 실제로 언제 유용할까

두 번째 요청의 `num_cached_tokens`가 0보다 훨씬 큰 값으로 나왔다면, `shared_prefix`에 해당하는 부분을 다시 계산하지 않고 그대로 재사용했다는 뜻입니다. (짧은 실험이라 시간 차이는 환경에 따라 크지 않게 느껴질 수 있지만, `num_cached_tokens`는 실제로 몇 개의 토큰이 재사용되었는지 보여주는 훨씬 직접적인 증거입니다.)

이 기능이 실제로 빛을 발하는 대표적인 상황:

- **긴 문서 Q&A**: 같은 매뉴얼/보고서를 반복해서 참고하며 여러 질문을 던지는 경우. 문서 부분을 한 번만 계산해두고 질문마다 재사용합니다.
- **멀티턴 대화(챗봇)**: 대화가 이어질수록 "지금까지의 대화 기록" 전체가 매번 앞부분에 그대로 포함됩니다. 매 턴마다 이 기록 전체를 다시 계산하지 않아도 됩니다.
- **동일한 시스템 프롬프트를 쓰는 다수의 사용자**: 서비스 전체가 같은 지침(시스템 프롬프트)으로 시작한다면, 그 부분은 사실상 한 번만 계산됩니다.

반대로, 질문마다 서로 겹치는 부분이 전혀 없거나(Part 8의 3개 프롬프트처럼), 생성 길이가 매우 길어서 decoding이 대부분의 시간을 차지하는 경우에는 Prefix Caching의 효과가 잘 드러나지 않습니다.

## Part 10. naive vs vLLM 비교 정리

Part 3(naive)과 Part 7(vLLM)에서 각각 측정한 시간을 나란히 정리해보겠습니다. (Part 3을 건너뛰었다면, 이 셀은 `naive_elapsed` 변수가 없다는 안내만 출력하고 넘어갑니다.)

In [ ]:
try:
    print(f"naive 방식 : {naive_elapsed:.2f}초")
    print(f"vLLM 방식  : {vllm_elapsed:.2f}초")
    print(f"속도 차이  : 약 {naive_elapsed / vllm_elapsed:.2f}배")
    print()
    print("참고: 프롬프트가 3개뿐이라 차이가 크게 느껴지지 않을 수 있습니다.")
    print("prompts 리스트를 30개, 300개로 늘려서 다시 실행해보면,")
    print("naive 방식은 시간이 거의 정비례해서 늘어나지만")
    print("vLLM은 continuous batching 덕분에 훨씬 완만하게 늘어나는 것을 확인할 수 있습니다.")
except NameError:
    print("naive_elapsed가 없습니다. Part 3(naive 추론) 셀을 먼저 실행해야 비교할 수 있습니다.")

## Part 11. (심화) kv_cache_dtype 옵션 정리

`kv_cache_dtype`에는 `"fp8"` 외에도 몇 가지 값을 지정할 수 있습니다. 참고용으로 정리합니다.

| 값 | 설명 |
|---|---|
| `"auto"` | 모델의 기본 정밀도 사용 (보통 fp16/bf16, 토큰당 2바이트). 별도 설정을 하지 않을 때의 기본값입니다. |
| `"fp8"` | 하드웨어에 맞춰 자동으로 세부 형식을 선택합니다. (CUDA에서는 사실상 `fp8_e4m3`와 동일) |
| `"fp8_e4m3"` | 8비트 부동소수점 중 지수(exponent) 4비트 + 가수(mantissa) 3비트 형식. 표현 범위보다 정밀도를 조금 더 우선합니다. |
| `"fp8_e5m2"` | 지수 5비트 + 가수 2비트 형식. 정밀도보다 더 넓은 표현 범위를 우선합니다. |

실습 코드에서는 `"fp8"`을 사용했는데, 이는 Part 5의 메모리 계산에서 본 것처럼 fp16 대비 KV 캐시 메모리를 절반으로 줄여줍니다. 처음 vLLM을 다룬다면 `"auto"`(기본값)로 먼저 익숙해진 뒤, 메모리가 부족해질 때 `"fp8"`을 시도해보는 순서를 권장합니다.

## Part 12. 자주 겪는 오류 (Troubleshooting)

| 증상 | 원인 | 해결 방법 |
|---|---|---|
| `CUDA out of memory` | `gpu_memory_utilization`이 너무 높거나, 다른 프로세스(예: Part 3의 transformers 모델)가 이미 GPU를 점유 중 | `gpu_memory_utilization`을 낮추기(예: 0.7), 더 작은 모델 사용, 커널 재시작으로 메모리 회수 |
| `403 Forbidden` / `you don't have access to this gated repo` | Gated 모델(예: Llama 3)에 라이선스 동의·로그인을 하지 않음 | Part 2-2의 Hugging Face 로그인 절차 수행 |
| `ModuleNotFoundError: No module named 'vllm'` | vLLM 미설치 | Part 1-2 설치 셀 재실행 후 커널 재시작 |
| Colab에서 GPU를 인식하지 못함 | 런타임 유형이 CPU로 설정됨 | `런타임 → 런타임 유형 변경 → 하드웨어 가속기: GPU` |
| 첫 `llm.generate()` 호출이 수 분씩 걸림 | 모델 다운로드 + 내부 초기화(웜업) 비용 (정상 현상) | 처음 한 번만 오래 걸리며, 이후 호출부터는 빨라집니다 |

## Part 13. 정리 + 스스로 실험해보기

### 오늘 배운 것 정리

- KV Cache는 계산 시간을 아끼지만, 그 대가로 메모리를 씁니다. vLLM은 이 메모리를 **똑똑하게 관리**하기 위한 서빙 엔진입니다.
- **PagedAttention**은 KV 캐시를 블록 단위로 나누어(OS의 가상 메모리 페이징과 유사) 메모리 낭비를 줄입니다.
- **Continuous Batching**은 요청이 끝나는 즉시 새 요청을 끼워 넣어 GPU를 놀리지 않습니다. (naive vs vLLM 비교로 체감)
- **temperature/top_p/top_k**는 다음 토큰을 고르는 확률분포를 조절하는 파라미터입니다.
- **Prefix Caching**은 공유되는 프롬프트 앞부분의 계산을 재사용합니다. (`num_cached_tokens`로 직접 확인 가능)
- **KV Cache Quantization**(`kv_cache_dtype="fp8"`)은 K/V 값을 더 적은 비트로 저장해 메모리를 절반으로 줄입니다.

### 스스로 해볼 실험

1. `temperature`를 0, 0.3, 1.0, 2.0으로 바꿔가며 Part 7을 다시 실행하고, 생성된 문장이 어떻게 달라지는지 비교해보세요.
2. Part 8의 3개 프롬프트 앞에 똑같은 문장 하나를 공통으로 붙여서, `num_cached_tokens`가 어떻게 달라지는지 확인해보세요.
3. `enable_prefix_caching=False`로 바꿔서 `llm`을 다시 만들고, Part 9 실험을 반복해 `num_cached_tokens`가 항상 0이 되는지 확인해보세요.
4. `kv_cache_dtype`을 `"auto"`와 `"fp8"`로 각각 바꿔가며 `nvidia-smi`로 GPU 메모리 사용량을 비교해보세요.
5. `prompts` 리스트를 3개에서 30개로 늘려서, naive와 vLLM의 시간 차이가 더 뚜렷해지는지 확인해보세요.

### 다음 실습 예고

이번 실습은 노트북 안에서 직접 파이썬 객체(`llm`)를 만들어 사용했습니다. 실제 서비스에서는 보통 vLLM을 **OpenAI 호환 API 서버**로 띄워두고(`vllm serve <모델명>`), 다른 애플리케이션에서 REST API처럼 호출하는 방식을 씁니다. 다음 실습 코드에서 이 서빙 방식을 다뤄보겠습니다.